# House Price Prediction
## Notebook 3 — Preprocessing and Feature Engineering
**Author: Tinashe Clifford Makandise**

## 1. Purpose
Define reproducible feature construction and unfitted preprocessing factories for model comparison. Notebook 2 found strong rank associations for quality, area and garage capacity, skewed prices, sparse neighbourhoods and four records requiring review. Those findings motivate candidates, not automatic exclusions.

This notebook reads **training data only**. It creates shared Python modules so later notebooks and the app use identical feature logic. No prediction model is selected or evaluated here.

### Prerequisites
Run Notebook 1 first. Save this file in `notebooks/`. Dependencies: `%pip install pandas numpy scikit-learn`. Python must be able to write `src/` and `reports/` within the project folder.

In [1]:
from pathlib import Path
import json
import sys
import importlib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import train_test_split
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string() if hasattr(value, "to_string") else value)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
train = pd.read_csv(PROJECT_ROOT / "data/processed/train.csv", keep_default_na=False, na_values=[""])
manifest = json.loads((PROJECT_ROOT / "reports/data_manifest.json").read_text(encoding="utf-8"))
assert set(train["PID"].astype(int)) == set(manifest["train_pids"])
assert len(train) == manifest["train_rows"]
assert not set(train["PID"].astype(int)).intersection(manifest["test_pids"])
print(f"Verified {len(train):,} training properties; test file not opened.")

Verified 2,344 training properties; test file not opened.


## 2. Feature contract
We compare a core app feature set with an expanded set adding quality and condition. Bedrooms and bathrooms refer to above-ground rooms. Garage capacity is car spaces in a garage, not all parking. Age is the difference between the historical sale year and build year. The historical demo must not silently substitute today's year.

Missing numeric measurements are retained for fold-specific median imputation. Zero bedrooms, bathrooms or garage spaces are retained: zero is not automatically missing. Invalid negative counts, nonpositive area, and infinite values cause an explicit error. A build date after the sale year is inconsistent: its derived age becomes missing for fold-specific imputation, while the row and original dates are retained. Fractional counts and implausible app inputs will receive additional user-input validation when the app is built.

Only neighbourhood is used categorically in this first version. Literal `NA` is not a valid neighbourhood and becomes unknown. Amenity absence codes in excluded columns are preserved in the raw partitions; no global replacement or imputation is applied to them. IDs, sale price, sale condition and other unused columns are excluded from model inputs.

In [2]:
schema = pd.DataFrame([
    ("Gr Liv Area", "Above-ground area", "sq ft", "core"),
    ("Bedroom AbvGr", "Above-ground bedrooms", "count", "core"),
    ("Full Bath", "Above-ground full bathrooms", "count", "core"),
    ("Half Bath", "Above-ground half bathrooms", "count", "core"),
    ("AgeAtSale", "Yr Sold minus Year Built", "years", "core"),
    ("Garage Cars", "Garage capacity", "cars", "core"),
    ("Neighborhood", "Location within Ames", "category", "core"),
    ("Overall Qual", "Material/finish quality", "1–10", "expanded"),
    ("Overall Cond", "Overall condition", "1–10", "expanded"),
], columns=["feature", "meaning", "unit", "feature_set"])
display(schema)

         feature                      meaning      unit feature_set
0    Gr Liv Area            Above-ground area     sq ft        core
1  Bedroom AbvGr        Above-ground bedrooms     count        core
2      Full Bath  Above-ground full bathrooms     count        core
3      Half Bath  Above-ground half bathrooms     count        core
4      AgeAtSale     Yr Sold minus Year Built     years        core
5    Garage Cars              Garage capacity      cars        core
6   Neighborhood         Location within Ames  category        core
7   Overall Qual      Material/finish quality      1–10    expanded
8   Overall Cond            Overall condition      1–10    expanded


## 3. Write reusable feature construction
The next cell creates `src/features.py`. It performs deterministic, row-level transformations only and learns no information from the target. The same function will run inside the final prediction pipeline. Module files are intentionally generated by this notebook so downloading and running it is enough to reproduce the shared code. Rerunning overwrites these project-specific generated files.

In [3]:
SRC_DIR = PROJECT_ROOT / "src"
SRC_DIR.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "__init__.py").touch()
feature_source = '"""Shared property feature construction. Author: Tinashe Clifford Makandise."""\nimport numpy as np\nimport pandas as pd\n\nCORE_NUMERIC = ["Gr Liv Area", "Bedroom AbvGr", "Full Bath", "Half Bath", "AgeAtSale", "Garage Cars"]\nEXPANDED_NUMERIC = CORE_NUMERIC + ["Overall Qual", "Overall Cond"]\nCATEGORICAL = ["Neighborhood"]\nRAW_CORE = ["Gr Liv Area", "Bedroom AbvGr", "Full Bath", "Half Bath", "Year Built", "Yr Sold", "Garage Cars", "Neighborhood"]\n\ndef build_features(frame):\n    """Derive age at sale and select supported inputs without learning statistics."""\n    if not isinstance(frame, pd.DataFrame):\n        raise TypeError("Pass a pandas DataFrame with named property columns.")\n    missing = sorted(set(RAW_CORE) - set(frame.columns))\n    if missing:\n        raise ValueError(f"Required columns missing: {missing}")\n    out = frame.copy()\n    numeric = [c for c in RAW_CORE if c != "Neighborhood"]\n    numeric += [c for c in ["Overall Qual", "Overall Cond"] if c in out.columns]\n    for col in numeric:\n        original = out[col]\n        converted = pd.to_numeric(original, errors="coerce")\n        invalid = original.notna() & converted.isna()\n        if invalid.any():\n            raise ValueError(f"Non-numeric values in {col}")\n        if np.isinf(converted).any():\n            raise ValueError(f"Infinite values in {col}")\n        out[col] = converted.astype(float)\n    out["AgeAtSale"] = (out["Yr Sold"] - out["Year Built"]).where(out["Yr Sold"] >= out["Year Built"])\n    nonnegative = ["Bedroom AbvGr", "Full Bath", "Half Bath", "Garage Cars"]\n    if out[nonnegative].lt(0).any().any() or out["Gr Liv Area"].le(0).any():\n        raise ValueError("Area must be positive; counts and age must be nonnegative.")\n    for col in ["Overall Qual", "Overall Cond"]:\n        if col in out and (~out[col].between(1, 10) & out[col].notna()).any():\n            raise ValueError(f"{col} must be between 1 and 10.")\n    out["Neighborhood"] = out["Neighborhood"].map(\n        lambda v: np.nan if pd.isna(v) or str(v).strip() in ("", "NA") else str(v).strip()\n    )\n    selected = CORE_NUMERIC + [c for c in ["Overall Qual", "Overall Cond"] if c in out] + CATEGORICAL\n    return out[selected]\n'
(SRC_DIR / "features.py").write_text(feature_source, encoding="utf-8")
print("Created src/features.py")

Created src/features.py


## 4. Build preprocessing factories
Numeric inputs use median imputation; linear models also use standardisation. Tree models will use the unscaled variant. Categorical values use a constant unknown label and one-hot encoding. An unseen neighbourhood produces an all-zero location block: prediction remains possible, but this is not evidence of reliable geographic transfer. The app should restrict choices to supported locations.

All one-hot columns are retained. Linear Regression can handle redundant columns numerically, but individual coefficients may be unstable; Ridge provides a regularised comparison. We will avoid interpreting a single location coefficient as a causal effect.

**Leakage control:** the factory returns an unfitted pipeline. In Notebook 4 the entire pipeline must be passed into cross-validation, so medians, scaling and category vocabularies are learned separately in each training fold. Do not pre-transform the full training dataset before cross-validation.

In [4]:
modeling_source = '"""Unfitted model-pipeline factories. Author: Tinashe Clifford Makandise."""\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler\nfrom .features import build_features, CORE_NUMERIC, EXPANDED_NUMERIC, CATEGORICAL\n\ndef make_preprocessor(expanded=False, scale_numeric=True):\n    numeric = EXPANDED_NUMERIC if expanded else CORE_NUMERIC\n    steps = [("impute", SimpleImputer(strategy="median"))]\n    if scale_numeric:\n        steps.append(("scale", StandardScaler()))\n    categorical = Pipeline([\n        ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),\n        ("encode", OneHotEncoder(handle_unknown="ignore", drop=None)),\n    ])\n    return ColumnTransformer([\n        ("numeric", Pipeline(steps), numeric),\n        ("categorical", categorical, CATEGORICAL),\n    ], remainder="drop")\n\ndef make_model_pipeline(estimator, expanded=False, scale_numeric=True):\n    return Pipeline([\n        ("features", FunctionTransformer(build_features, validate=False)),\n        ("preprocess", make_preprocessor(expanded, scale_numeric)),\n        ("model", estimator),\n    ])\n'
(SRC_DIR / "modeling.py").write_text(modeling_source, encoding="utf-8")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
importlib.invalidate_caches()
import src.features as features
import src.modeling as modeling
importlib.reload(features)
importlib.reload(modeling)
print("Created src/modeling.py and loaded shared functions.")

Created src/modeling.py and loaded shared functions.


## 5. Validate feature construction
This check ensures feature order and values are consistent and that identifiers and the target do not reach preprocessing. No rows are removed, including the previously flagged large houses. Exclusion will require a documented reason; retaining them preserves the evaluation population.

In [5]:
X = train.drop(columns="SalePrice")
y = train["SalePrice"].copy()
engineered = features.build_features(X)
assert len(engineered) == len(train)
assert not {"SalePrice", "PID", "Order", "Yr Sold", "Year Built"}.intersection(engineered.columns)
expected_age = (train["Yr Sold"] - train["Year Built"]).where(train["Yr Sold"] >= train["Year Built"])
np.testing.assert_allclose(engineered["AgeAtSale"], expected_age, equal_nan=True)
display(engineered.head())
print("Missing values among candidate features:")
display(engineered.isna().sum().to_frame("missing_count"))

   Gr Liv Area  Bedroom AbvGr  Full Bath  Half Bath  AgeAtSale  Garage Cars  Overall Qual  Overall Cond Neighborhood
0       1656.0            3.0        1.0        0.0       50.0          2.0           6.0           5.0        NAmes
1        896.0            2.0        1.0        0.0       49.0          1.0           5.0           6.0        NAmes
2       1329.0            3.0        1.0        1.0       52.0          1.0           6.0           6.0        NAmes
3       2110.0            3.0        2.0        1.0       42.0          2.0           7.0           5.0        NAmes
4       1629.0            3.0        2.0        1.0       13.0          2.0           5.0           5.0      Gilbert
Missing values among candidate features:
               missing_count
Gr Liv Area                0
Bedroom AbvGr              0
Full Bath                  0
Half Bath                  0
AgeAtSale                  1
Garage Cars                1
Overall Qual               0
Overall Cond             

## 6. Demonstrate fitting on an internal training subset
A small internal split is used solely to check transformation behaviour. Its validation part is not used to compare prediction models. The demonstration preprocessor learns only from its fitting subset and is discarded afterward. No fitted preprocessor is exported for model comparison.

In [6]:
X_demo_fit, X_demo_check = train_test_split(X, test_size=.20, random_state=42)
fit_features = features.build_features(X_demo_fit)
check_features = features.build_features(X_demo_check)
preview = modeling.make_preprocessor(expanded=True, scale_numeric=True)
fit_matrix = preview.fit_transform(fit_features)
check_matrix = preview.transform(check_features)
assert fit_matrix.shape[1] == check_matrix.shape[1]
assert fit_matrix.shape[0] == len(X_demo_fit)
# Work with sparse or dense output without assuming encoder output format.
def finite_matrix(matrix):
    values = matrix.data if hasattr(matrix, "toarray") else np.asarray(matrix)
    return np.isfinite(values).all()
assert finite_matrix(fit_matrix) and finite_matrix(check_matrix)
print("Internal fit shape:", fit_matrix.shape, "check shape:", check_matrix.shape)
print("Feature names:", preview.get_feature_names_out().tolist())
assert not hasattr(clone(preview), "transformers_")
print("Cloning returns an unfitted preprocessor for independent folds.")

Internal fit shape: (1875, 36) check shape: (469, 36)
Feature names: ['numeric__Gr Liv Area', 'numeric__Bedroom AbvGr', 'numeric__Full Bath', 'numeric__Half Bath', 'numeric__AgeAtSale', 'numeric__Garage Cars', 'numeric__Overall Qual', 'numeric__Overall Cond', 'categorical__Neighborhood_Blmngtn', 'categorical__Neighborhood_Blueste', 'categorical__Neighborhood_BrDale', 'categorical__Neighborhood_BrkSide', 'categorical__Neighborhood_ClearCr', 'categorical__Neighborhood_CollgCr', 'categorical__Neighborhood_Crawfor', 'categorical__Neighborhood_Edwards', 'categorical__Neighborhood_Gilbert', 'categorical__Neighborhood_Greens', 'categorical__Neighborhood_GrnHill', 'categorical__Neighborhood_IDOTRR', 'categorical__Neighborhood_Landmrk', 'categorical__Neighborhood_MeadowV', 'categorical__Neighborhood_Mitchel', 'categorical__Neighborhood_NAmes', 'categorical__Neighborhood_NPkVill', 'categorical__Neighborhood_NWAmes', 'categorical__Neighborhood_NoRidge', 'categorical__Neighborhood_NridgHt', 'categ

## 7. Check missing and unseen inputs
These are functional checks, not performance estimates. A missing numeric measurement must be imputed without producing non-finite output. Unknown location must preserve matrix width. A malformed property should fail clearly. An all-zero encoded location block is an engineering fallback, not a confidence statement.

In [7]:
probe = X_demo_check.iloc[[0]].copy()
probe.loc[:, "Neighborhood"] = "UNSEEN_LOCATION_FOR_CHECK"
probe.loc[:, "Gr Liv Area"] = np.nan
probe_matrix = preview.transform(features.build_features(probe))
assert probe_matrix.shape[1] == fit_matrix.shape[1] and finite_matrix(probe_matrix)
bad = X_demo_check.iloc[[0]].copy()
bad.loc[:, "Gr Liv Area"] = -10
try:
    features.build_features(bad)
except ValueError as exc:
    print("Invalid-input check passed:", exc)
else:
    raise AssertionError("Negative area was not rejected")
print("Missing numeric and unseen-category transformation checks passed.")

<notebook>:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
Invalid-input check passed: Area must be positive; counts and age must be nonnegative.
Missing numeric and unseen-category transformation checks passed.


## 8. Prepare model-ready factories, not pre-fitted data
The full pipeline places deterministic features before fold-specific preprocessing. This cell assembles and inspects a model pipeline without fitting it. Notebook 4 will compare core/expanded feature sets and model families through cross-validation. Random Forest will use `scale_numeric=False`.

A log-target alternative may use `TransformedTargetRegressor` around the full model pipeline with `np.log1p` and `np.expm1`; its predictions must be compared in USD. Predictions from inverse log transforms can have retransformation bias, so we will assess actual validation errors rather than assume an improvement.

In [8]:
from sklearn.linear_model import Ridge
candidate = modeling.make_model_pipeline(Ridge(alpha=1.0), expanded=True, scale_numeric=True)
assert not hasattr(candidate.named_steps["preprocess"], "transformers_")
assert not hasattr(candidate.named_steps["model"], "coef_")
print(candidate)
print("Unfitted end-to-end pipeline prepared; no model trained.")

Pipeline(steps=[('features',
                 FunctionTransformer(func=<function build_features at 0x7fb6113ea480>)),
                ('preprocess',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scale',
                                                                   StandardScaler())]),
                                                  ['Gr Liv Area',
                                                   'Bedroom AbvGr', 'Full Bath',
                                                   'Half Bath', 'AgeAtSale',
                                                   'Garage Cars',
                                                   'Overall Qual',
                                                   'Overall Cond']),
                     

## 9. Export the handover specification
The source partitions remain unchanged. We export the input contract and decisions, while the modules contain reusable factories. Numerical bounds in the table below describe observed training coverage, not universal physical limits or hard deployment rules. No encoded matrix or fitted imputer is saved.

In [9]:
REPORTS_DIR = PROJECT_ROOT / "reports"
schema.to_csv(REPORTS_DIR / "feature_schema.csv", index=False)
coverage = engineered[features.EXPANDED_NUMERIC].agg(["min", "max", "median"]).T
coverage.to_csv(REPORTS_DIR / "training_feature_coverage.csv", index_label="feature")
specification = {
    "author": "Tinashe Clifford Makandise",
    "core_numeric": features.CORE_NUMERIC,
    "expanded_numeric": features.EXPANDED_NUMERIC,
    "categorical": features.CATEGORICAL,
    "raw_required_core": features.RAW_CORE,
    "age_definition": "Yr Sold - Year Built; negative derived ages become missing",
    "numeric_imputation": "median fitted within each training fold",
    "categorical_imputation": "Unknown constant",
    "unknown_category": "ignore during one-hot encoding; flag unsupported locations in app",
    "row_exclusions": "none",
    "test_data_used": False,
    "target": "SalePrice", "currency": "historical USD",
}
(REPORTS_DIR / "preprocessing_specification.json").write_text(json.dumps(specification, indent=2), encoding="utf-8")
display(coverage)
del preview, fit_matrix, check_matrix, probe_matrix
print("Saved feature schema, training coverage and preprocessing specification.")

                 min     max  median
Gr Liv Area    334.0  5642.0  1436.5
Bedroom AbvGr    0.0     8.0     3.0
Full Bath        0.0     4.0     2.0
Half Bath        0.0     2.0     0.0
AgeAtSale        0.0   136.0    36.0
Garage Cars      0.0     4.0     2.0
Overall Qual     1.0    10.0     6.0
Overall Cond     1.0     9.0     5.0
Saved feature schema, training coverage and preprocessing specification.


## 10. Conclusions and next stage
- Deterministic feature construction is shared through `src/features.py`.
- `src/modeling.py` constructs unfitted pipelines for numeric imputation, optional scaling, categorical encoding and an estimator.
- Functional checks verified age construction, identifier exclusion, finite transformed values, stable width for unseen locations and rejection of negative area.
- No properties were excluded, no predictive model was fitted and no test data was opened.

**Next notebook:** compare a median-price baseline, Linear Regression, Ridge and Random Forest using training-only cross-validation. Select configurations using MAE in USD, with RMSE and R² as supporting metrics. The final holdout is reserved for Notebook 5.

When sharing or deploying a saved final pipeline later, keep the `src` package alongside the application because the pipeline references `src.features.build_features`.